# Finding the animal, properly

Notebook 09 tried to locate animals by subtracting each camera's background. Three designs,
three failures — min/max boxes, coordinate percentiles, peak-anchored spread. The last
attempt had a measurably *better* background (day and night estimated separately, own-camera
residual 1.86x another camera's) and produced *worse* boxes: median side 0.93 of the frame,
46% covering everything.

That is not a tuning problem. Motion residual is the wrong signal. A camera trap's frame
differs from its own median for many reasons — a branch moved, the sun went behind a cloud,
the IR flash fired unevenly, the grass grew over three months — and "changed" is simply not
the same question as "is an animal".

So this notebook throws that out and asks the question directly, two ways.

### 1. MegaDetector

A real object detector, trained by Microsoft's AI for Good lab on millions of camera-trap
images from exactly this kind of deployment. It answers "where is the animal" because that is
what it was trained to answer. This is what the camera-trap field actually uses, and I should
have reached for it two notebooks ago instead of hand-rolling a third residual heuristic.

### 2. CLIP-scan, if the install misbehaves

A second new method that needs nothing installed. BioCLIP already scores how well an image
matches the text "a camera trap photo of *Leopardus pardalis*". So: cut the frame into a
multi-scale grid of candidate windows, score every one against the animal prompts minus the
empty-scene prompts, and keep the window that wins. The backbone becomes its own detector.
Semantic rather than motion-based — a different question from the one 09 kept asking.

### What makes this checkable, finally

Both detectors output a **confidence**, and the dataset has 600 `empty` images per eval split.
A detector that works must be less confident on empty frames than on animal frames, and
section 6 measures exactly that as an AUC before a single crop is embedded. Notebook 09 never
had this check — its residual strength was never tested against a label. This time the
detector is validated against ground truth on 300 images in under a minute, and the notebook
refuses to spend the long pass if that number says the detector is blind.

## 1. Setup

In [1]:
import io
import json
import time
import zipfile
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams["figure.figsize"] = (11, 4)
print("torch:", torch.__version__, "| device:", DEVICE)

torch: 2.11.0+cu128 | device: cuda


In [2]:
for package, module in [("open_clip_torch", "open_clip"), ("xgboost", "xgboost")]:
    try:
        __import__(module)
    except ModuleNotFoundError:
        print(f"installing {package} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import open_clip
import xgboost
from xgboost import XGBClassifier

ACCEL = ({"tree_method": "hist", "device": "cuda" if torch.cuda.is_available() else "cpu"}
         if int(xgboost.__version__.split(".")[0]) >= 2
         else {"tree_method": "gpu_hist" if torch.cuda.is_available() else "hist"})
print("open_clip:", open_clip.__version__, "| xgboost:", xgboost.__version__)

installing open_clip_torch ...
open_clip: 3.3.0 | xgboost: 3.4.1


## 2. Configuration

`DETECTOR` is the whole point of this notebook. Everything else matches 09 so the comparison
is clean.

In [3]:
from google.colab import drive
drive.mount("/content/drive")

ZIP_PATH = Path("/content/drive/MyDrive/IWildCam v2.0 2020 WILDS Dataset.zip")
CACHE_DIR = Path("/content/drive/MyDrive/iwildcam_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 192          # the cached frames, used for the frame stream only
N_CLASSES = 10
CAP_TRAIN = 3000
CAP_EVAL = 600

HOLDOUT_FRAC = 0.15
PCA_DIMS = 128
ES_FRAC = 0.15

BACKBONE = "bioclip"
DETECTOR = "megadetector"   # "megadetector" or "clipscan"
WORK_MAX = 1333         # originals are resized to this longest side before detection
CROP_PAD = 0.15         # detector boxes are tight already, so less padding than 09 needed
CROP_MIN = 0.08
MIN_CONF = 0.15         # below this the detector is not believed; the frame is used whole
TTA_FLIP = True
CHUNK = 500             # images per checkpoint on the long pass

Mounted at /content/drive


## 3. Data

Identical to 06-09: same ten classes, same sampling, same official splits, same seed, same
three groups of cameras.

In [4]:
ROOT = "iwildcam_v2.0"


class IWildCamSource:
    "Reads metadata and image bytes from the WILDS ZIP, and can reopen after a mount drop."

    def __init__(self, path):
        self.path = Path(path)
        self.is_zip = self.path.suffix.lower() == ".zip"
        self._open()

    def _open(self):
        if self.is_zip:
            self.zf = zipfile.ZipFile(self.path)
            self.offsets = {i.filename: i.header_offset for i in self.zf.infolist()}
        else:
            self.dir = self.path if self.path.name == ROOT else self.path / ROOT

    def reopen(self):
        try:
            self.zf.close()
        except Exception:
            pass
        self._open()

    def read(self, rel):
        if self.is_zip:
            return self.zf.read(f"{ROOT}/{rel}")
        return (self.dir / rel).read_bytes()

    def read_order(self, rel):
        return self.offsets.get(f"{ROOT}/{rel}", 0) if self.is_zip else 0


source = IWildCamSource(ZIP_PATH)
metadata = pd.read_csv(io.BytesIO(source.read("metadata.csv")), index_col=0)
categories = pd.read_csv(io.BytesIO(source.read("categories.csv")))

COMMON = {
    "empty": "empty (false trigger)",
    "meleagris ocellata": "ocellated turkey",
    "crax rubra": "great curassow",
    "tayassu pecari": "white-lipped peccary",
    "cephalophus nigrifrons": "black-fronted duiker",
    "dasyprocta punctata": "agouti",
    "leopardus pardalis": "ocelot",
    "mazama temama": "red brocket deer",
    "puma concolor": "puma",
    "urocyon cinereoargenteus": "gray fox",
}

scientific = dict(zip(categories.y, categories.name))
counts = metadata[metadata.split == "train"].y.value_counts()
top_labels = counts.head(N_CLASSES).index.tolist()
CLASS_NAMES = [COMMON.get(scientific[y], scientific[y]) for y in top_labels]
SCIENTIFIC_NAMES = [scientific[y] for y in top_labels]
label_to_index = {y: i for i, y in enumerate(top_labels)}
EMPTY_INDEX = CLASS_NAMES.index("empty (false trigger)")


def sample_split(split, cap, seed=SEED):
    "Take up to `cap` images per class from one official split, spread across sequences."
    rows = metadata[(metadata.split == split) & (metadata.y.isin(top_labels))]
    rng = np.random.default_rng(seed)
    kept = []
    for y in top_labels:
        cls = rows[rows.y == y]
        groups = list(cls.groupby("seq_id").indices.values())
        rng.shuffle(groups)
        order, depth = [], 0
        while len(order) < len(cls):
            taken = [cls.index[g[depth]] for g in groups if depth < len(g)]
            if not taken:
                break
            order.extend(taken)
            depth += 1
        kept.extend(order[:cap])
    return metadata.loc[kept]


SPLITS = {
    "train":   sample_split("train", CAP_TRAIN),
    "id_val":  sample_split("id_val", CAP_EVAL),
    "id_test": sample_split("id_test", CAP_EVAL),
    "ood_val": sample_split("val", CAP_EVAL),
}
LABELS = {n: np.array([label_to_index[v] for v in f.y], dtype=np.int64)
          for n, f in SPLITS.items()}


def camera_holdout(frame, target_frac=HOLDOUT_FRAC, seed=SEED):
    "Split a frame by CAMERA, repairing the result so every class appears on both sides."
    rng = np.random.default_rng(seed)
    sizes = frame.location_remapped.value_counts()
    held, n = [], 0
    for cam in rng.permutation(sizes.index.to_numpy()):
        if n >= target_frac * len(frame):
            break
        held.append(cam)
        n += sizes[cam]

    for y in frame.y.unique():
        if (frame[frame.location_remapped.isin(held)].y == y).sum() == 0:
            held.append(frame[(frame.y == y)].location_remapped.value_counts().index[0])
        if (frame[~frame.location_remapped.isin(held)].y == y).sum() == 0:
            held = [c for c in held
                    if (frame[frame.location_remapped == c].y == y).mean() < 1.0]

    mask = frame.location_remapped.isin(held).to_numpy()
    return ~mask, mask


train_frame = SPLITS["train"]
FIT_MASK, SEL_MASK = camera_holdout(train_frame)
INNER_FIT, INNER_ES = camera_holdout(train_frame[FIT_MASK], ES_FRAC, seed=SEED + 1)
y_fit = LABELS["train"][FIT_MASK]
y_sel = LABELS["train"][SEL_MASK]

print(f"{'split':>8} {'images':>8} {'sequences':>10} {'cameras':>9} {'empty':>7}")
for name, frame in SPLITS.items():
    print(f"{name:>8} {len(frame):>8,} {frame.seq_id.nunique():>10,} "
          f"{frame.location_remapped.nunique():>9} "
          f"{(LABELS[name] == EMPTY_INDEX).sum():>7,}")
print(f"\nfit {FIT_MASK.sum():,} / selection {SEL_MASK.sum():,} images, "
      f"{train_frame[SEL_MASK].location_remapped.nunique()} selection cameras")

   split   images  sequences   cameras   empty
   train   29,740      8,773       208   3,000
  id_val    2,377        872       107     600
 id_test    3,420        982       137     600
 ood_val    4,361      1,444        29     600

fit 25,211 / selection 4,529 images, 36 selection cameras


## 4. The backbone

BioCLIP, unchanged from 08 and 09. It embeds crops, and under `clipscan` it also does the
detecting.

In [5]:
def load_backbone():
    for label, name, pretrained in [("BioCLIP", "hf-hub:imageomics/bioclip", None),
                                    ("OpenCLIP ViT-B/16", "ViT-B-16", "laion2b_s34b_b88k")]:
        try:
            model, _, preprocess = open_clip.create_model_and_transforms(
                name, pretrained=pretrained)
            tokenizer = open_clip.get_tokenizer(name)
            break
        except Exception as exc:
            print(f"could not load {label}: {type(exc).__name__}: {exc}")
    else:
        raise RuntimeError("no backbone could be loaded")

    norm = [t for t in preprocess.transforms if t.__class__.__name__ == "Normalize"][0]
    size = model.visual.image_size
    size = size[0] if isinstance(size, (tuple, list)) else size
    return (label, model.eval().to(DEVICE), tokenizer, size,
            torch.tensor(norm.mean, device=DEVICE).view(1, 3, 1, 1),
            torch.tensor(norm.std, device=DEVICE).view(1, 3, 1, 1))


BACKBONE_LABEL, MODEL, TOKENIZER, SIZE, MEAN, STD = load_backbone()
print(f"backbone: {BACKBONE_LABEL} at {SIZE}px")


@torch.no_grad()
def embed_tensor(X, batch_size=256, flip=TTA_FLIP):
    "Embed a uint8 (N,3,H,W) tensor: resize, normalize, L2-normalize, mirror-average."
    out = []
    for i in range(0, len(X), batch_size):
        xb = X[i:i + batch_size].to(DEVICE, non_blocking=True).float() / 255
        if xb.shape[-1] != SIZE or xb.shape[-2] != SIZE:
            xb = F.interpolate(xb, size=(SIZE, SIZE), mode="bicubic",
                               align_corners=False, antialias=True).clamp(0, 1)
        xb = (xb - MEAN) / STD
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            feats = MODEL.encode_image(xb).float()
            if flip:
                feats = feats + MODEL.encode_image(torch.flip(xb, dims=[3])).float()
        out.append(F.normalize(feats.float(), dim=-1).cpu())
    return torch.cat(out).numpy()


def class_prompts(i):
    if i == EMPTY_INDEX:
        return ["a camera trap photo of an empty scene with no animal",
                "an empty forest with no animals present",
                "a photo of vegetation and bare ground, no animal"]
    sci, common = SCIENTIFIC_NAMES[i], CLASS_NAMES[i]
    return [f"a camera trap photo of {sci}",
            f"a photo of {sci}, commonly known as {common}",
            f"a camera trap photo of a {common}"]


@torch.no_grad()
def text_weights():
    "One L2-normalized text embedding per class, averaged over its prompts."
    rows = []
    for i in range(N_CLASSES):
        tokens = TOKENIZER(class_prompts(i)).to(DEVICE)
        text = F.normalize(MODEL.encode_text(tokens).float(), dim=-1).mean(dim=0)
        rows.append(F.normalize(text, dim=0))
    return torch.stack(rows)


TEXT = text_weights()
print(f"text weights: {tuple(TEXT.shape)}")

open_clip_config.json:   0%|          | 0.00/469 [00:00<?, ?B/s]

open_clip_pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  599MB            

open_clip_pytorch_model.bin: downloading bytes:           |  0.00B            

backbone: BioCLIP at 224px
text weights: (10, 512)


## 5. Two detectors

Both take a full-resolution image and return one normalized box plus a confidence. Nothing
downstream knows which one produced it.

**MegaDetector** is an object detector trained on camera-trap imagery; the box with the
highest animal confidence wins.

**CLIP-scan** cuts the frame into 14 candidate windows — the whole frame, four halves at 60%,
nine tiles at 40% — embeds all of them, and scores each by how much more it looks like an
animal than like an empty scene, under the same prompts section 4 built. Confidence is that
margin. No training, no download, and a completely different question from motion.

In [6]:
def load_megadetector():
    "MegaDetector via PytorchWildlife, trying the current versions in turn."
    try:
        import PytorchWildlife
    except ModuleNotFoundError:
        print("installing PytorchWildlife ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PytorchWildlife"],
                       check=True)
    from PytorchWildlife.models import detection as pw_detection

    attempts = [("MegaDetectorV6", dict(version="MDV6-yolov10-c")),
                ("MegaDetectorV6", dict(version="MDV6-yolov9-c")),
                ("MegaDetectorV6", {}),
                ("MegaDetectorV5", {})]
    for cls_name, kwargs in attempts:
        cls = getattr(pw_detection, cls_name, None)
        if cls is None:
            continue
        try:
            model = cls(device=str(DEVICE), pretrained=True, **kwargs)
            print(f"loaded {cls_name} {kwargs.get('version', '')}".strip())
            return model
        except Exception as exc:
            print(f"  {cls_name} {kwargs.get('version', '')}: "
                  f"{type(exc).__name__}: {str(exc)[:120]}")
    raise RuntimeError("MegaDetector could not be loaded")


def _unpack(result):
    "PytorchWildlife returns a dict around a supervision Detections; normalize it."
    det = result.get("detections", result) if isinstance(result, dict) else result
    xyxy = np.asarray(getattr(det, "xyxy", np.empty((0, 4))), dtype=float).reshape(-1, 4)
    conf = np.asarray(getattr(det, "confidence", np.empty(0)), dtype=float).reshape(-1)
    raw = getattr(det, "class_id", None)
    cls = (np.asarray(raw, dtype=int).reshape(-1)
           if raw is not None and len(np.asarray(raw).reshape(-1)) == len(conf)
           else np.zeros(len(conf), dtype=int))
    return xyxy, conf, cls


def megadetector_batch(arrays, model, animal_class=0):
    "Highest-confidence animal box per image, normalized to (x0, y0, x1, y1) in [0, 1]."
    boxes = np.tile(np.array([0.0, 0.0, 1.0, 1.0], np.float32), (len(arrays), 1))
    confs = np.zeros(len(arrays), np.float32)

    for j, arr in enumerate(arrays):
        try:
            result = model.single_image_detection(arr)
        except TypeError:
            result = model.single_image_detection(arr, img_path=f"{j}.jpg")
        xyxy, conf, cls = _unpack(result)
        keep = (cls == animal_class) & np.isfinite(conf)
        if not keep.any():
            continue
        best = np.argmax(np.where(keep, conf, -1))
        h, w = arr.shape[:2]
        x0, y0, x1, y1 = xyxy[best]
        boxes[j] = [x0 / w, y0 / h, x1 / w, y1 / h]
        confs[j] = conf[best]
    return np.clip(boxes, 0, 1), confs


CLIPSCAN_GRID = [(1, 1.00), (2, 0.60), (3, 0.40)]


def _windows():
    "Candidate windows as normalized (x0, y0, x1, y1): whole frame, halves, thirds."
    out = []
    for n, side in CLIPSCAN_GRID:
        step = (1.0 - side) / max(n - 1, 1) if n > 1 else 0.0
        for iy in range(n):
            for ix in range(n):
                x0, y0 = ix * step, iy * step
                out.append((x0, y0, x0 + side, y0 + side))
    return np.array(out, dtype=np.float32)


WINDOWS = _windows()


@torch.no_grad()
def clipscan_batch(arrays, model=None):
    """Score every candidate window by animal-minus-empty similarity; keep the best.

    The backbone is the detector: a window that contains an ocelot matches the ocelot
    prompts more than it matches 'an empty scene with no animal', and a window of bare
    foliage does the opposite.
    """
    crops = np.empty((len(arrays) * len(WINDOWS), SIZE, SIZE, 3), dtype=np.uint8)
    for j, arr in enumerate(arrays):
        for k, box in enumerate(WINDOWS):
            crops[j * len(WINDOWS) + k] = crop_from_array(arr, box, SIZE)

    feats = embed_tensor(torch.from_numpy(crops).permute(0, 3, 1, 2).contiguous(),
                         flip=False)
    scores = torch.from_numpy(feats).to(DEVICE) @ TEXT.T          # (B*K, classes)
    animal = torch.cat([scores[:, :EMPTY_INDEX], scores[:, EMPTY_INDEX + 1:]], dim=1)
    margin = (animal.max(dim=1).values - scores[:, EMPTY_INDEX]).view(len(arrays), -1)

    best = margin.argmax(dim=1).cpu().numpy()
    return WINDOWS[best].copy(), margin.max(dim=1).values.cpu().numpy()

### 5.1 Reading, resizing and cropping

Originals are read once, resized so the longest side is `WORK_MAX`, detected on, and cropped
from — all in one pass, so each JPEG crosses the Drive mount exactly once. Reads retry and
remount, because that mount is what crashed notebook 07.

In [7]:
def crop_from_array(arr, box, size):
    "Crop a HWC uint8 array to a normalized box and resize to size x size."
    h, w = arr.shape[:2]
    x0, y0, x1, y1 = box
    x0, x1 = int(x0 * w), int(np.ceil(x1 * w))
    y0, y1 = int(y0 * h), int(np.ceil(y1 * h))
    x0, y0 = max(x0, 0), max(y0, 0)
    x1, y1 = min(max(x1, x0 + 2), w), min(max(y1, y0 + 2), h)
    return np.asarray(Image.fromarray(arr[y0:y1, x0:x1]).resize((size, size),
                                                                Image.BICUBIC))


def pad_box(box, pad=CROP_PAD, min_side=CROP_MIN):
    "Pad, square up and clamp a normalized box."
    x0, y0, x1, y1 = box
    if not np.isfinite([x0, y0, x1, y1]).all() or x1 <= x0 or y1 <= y0:
        return np.array([0.0, 0.0, 1.0, 1.0], np.float32)
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    side = min(max(max(x1 - x0, y1 - y0) * (1 + 2 * pad), min_side), 1.0)
    cx, cy = np.clip(cx, side / 2, 1 - side / 2), np.clip(cy, side / 2, 1 - side / 2)
    return np.array([cx - side / 2, cy - side / 2,
                     cx + side / 2, cy + side / 2], np.float32)


def read_original(rel, attempts=4):
    "Read one file from the ZIP, surviving a FUSE mount that drops mid-run."
    for k in range(attempts):
        try:
            return source.read(rel)
        except (OSError, zipfile.BadZipFile, KeyError) as exc:
            if k == attempts - 1:
                return None
            print(f"    read failed ({type(exc).__name__}), remounting ...")
            try:
                drive.mount("/content/drive", force_remount=True)
            except Exception:
                pass
            time.sleep(2 ** k)
            source.reopen()
    return None


def load_working(rel, max_side=WORK_MAX):
    "Read one original and resize so its longest side is at most `max_side`."
    raw = read_original(rel)
    if raw is None:
        return None
    try:
        img = Image.open(io.BytesIO(raw)).convert("RGB")
    except Exception:
        return None
    if max(img.size) > max_side:
        scale = max_side / max(img.size)
        img = img.resize((max(int(img.width * scale), 1), max(int(img.height * scale), 1)),
                         Image.BILINEAR)
    return np.asarray(img)

## 6. Does the detector see animals? A go / no-go on 300 images

The check notebook 09 never had. 150 `empty` frames and 150 animal frames, drawn from the
unseen-camera split so nothing here is a home game. If the detector works, its confidence is
higher on animal frames than on empty ones, and the AUC of that separation says by how much.

**0.5 is blind.** Below about 0.65 the detector is not worth cropping with, and the notebook
says so rather than spending an hour finding out the expensive way.

In [8]:
def roc_auc(scores, positive):
    "AUC via rank-sum; positive is a boolean mask."
    order = np.argsort(scores)
    ranks = np.empty(len(scores), float)
    ranks[order] = np.arange(1, len(scores) + 1)
    n_pos, n_neg = positive.sum(), (~positive).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    return (ranks[positive].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


DETECT = None
if DETECTOR == "megadetector":
    try:
        _md = load_megadetector()
        DETECT = lambda arrays: megadetector_batch(arrays, _md)
    except Exception as exc:
        print(f"\nMegaDetector unavailable ({type(exc).__name__}: {str(exc)[:100]})")
        print("falling back to clipscan")
        DETECTOR = "clipscan"
if DETECT is None:
    DETECT = clipscan_batch
print(f"detector: {DETECTOR}\n")


rng = np.random.default_rng(SEED)
probe_frame = SPLITS["ood_val"]
probe_y = LABELS["ood_val"]
pick = np.concatenate([
    rng.choice(np.flatnonzero(probe_y == EMPTY_INDEX), 150, replace=False),
    rng.choice(np.flatnonzero(probe_y != EMPTY_INDEX), 150, replace=False)])

names = probe_frame.filename.tolist()
probe_arrays, probe_ok = [], []
for i in pick:
    arr = load_working(f"train/{names[i]}")
    if arr is not None:
        probe_arrays.append(arr)
        probe_ok.append(i)
print(f"read {len(probe_arrays)}/{len(pick)} probe originals "
      f"at {probe_arrays[0].shape if probe_arrays else '-'}")

start = time.perf_counter()
probe_boxes, probe_conf = DETECT(probe_arrays)
elapsed = time.perf_counter() - start

is_empty = probe_y[np.array(probe_ok)] == EMPTY_INDEX
auc = roc_auc(probe_conf, ~is_empty)
sides = np.array([max(b[2] - b[0], b[3] - b[1]) for b in probe_boxes])

print(f"\n{elapsed / max(len(probe_arrays), 1) * 1000:.0f} ms per image "
      f"-> about {elapsed / max(len(probe_arrays), 1) * 39898 / 60:.0f} min for all 39,898")
print(f"mean confidence: animal {probe_conf[~is_empty].mean():.3f}, "
      f"empty {probe_conf[is_empty].mean():.3f}")
print(f"median box side: animal {np.median(sides[~is_empty]):.2f}, "
      f"empty {np.median(sides[is_empty]):.2f}")
print(f"\nAUC separating animal from empty: {auc:.3f}")
if auc >= 0.65:
    print("the detector sees animals -> run section 7")
else:
    print("THE DETECTOR IS NOT SEEING ANIMALS -> do not run section 7.")
    print("if this was clipscan, try DETECTOR = 'megadetector'; if it was MegaDetector,")
    print("check that load_working returned real images above")

installing PytorchWildlife ...
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


Ultralytics 8.4.159 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv10n summary (fused): 99 layers, 2,265,753 parameters, 0 gradients
loaded MegaDetectorV6 MDV6-yolov10-c
detector: megadetector

read 300/300 probe originals at (448, 597, 3)

0: 1280x1280 (no detections), 20.4ms
Speed: 37.8ms preprocess, 20.4ms inference, 2.3ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 20.4ms
Speed: 8.8ms preprocess, 20.4ms inference, 0.9ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 20.4ms
Speed: 5.5ms preprocess, 20.4ms inference, 0.8ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 20.4ms
Speed: 6.0ms preprocess, 20.4ms inference, 0.8ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 20.4ms
Speed: 5.4ms preprocess, 20.4ms inference, 0.8ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 16.8ms
Speed: 5.4ms p

In [9]:
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
show_animal = [j for j, e in enumerate(is_empty) if not e][:6]
show_empty = [j for j, e in enumerate(is_empty) if e][:6]

for row, (group, title) in enumerate([(show_animal, "animal"), (show_empty, "empty")]):
    for col, j in enumerate(group):
        arr = probe_arrays[j]
        h, w = arr.shape[:2]
        x0, y0, x1, y1 = probe_boxes[j] * np.array([w, h, w, h])
        axes[row, col].imshow(arr)
        axes[row, col].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                                               color="yellow", lw=2))
        axes[row, col].set_title(f"{title} conf {probe_conf[j]:.2f}", fontsize=8)
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

## 7. The long pass

Read, detect, crop, embed — once per image, checkpointed to Drive every `CHUNK` images. If
the cell dies, re-run it and it resumes from where it stopped.

The frame stream is reused from notebook 08's cache, so only the crop stream is computed here.

In [10]:
def detect_and_embed(split, dim):
    "One pass over a split's originals; returns (crop embeddings, boxes, confidences)."
    tag = f"{DETECTOR}_{BACKBONE}"
    store = CACHE_DIR / f"det_{tag}_{split}.npy"
    meta_path = CACHE_DIR / f"det_{tag}_{split}_boxes.npy"
    state_path = CACHE_DIR / f"det_{tag}_{split}.json"
    n = len(SPLITS[split])
    names = SPLITS[split].filename.tolist()

    def open_store(path, shape):
        if path.exists():
            arr = np.lib.format.open_memmap(path, mode="r+")
            if arr.shape == shape:
                return arr
        return np.lib.format.open_memmap(path, mode="w+", dtype=np.float32, shape=shape)

    emb = open_store(store, (n, dim))
    boxconf = open_store(meta_path, (n, 5))
    done = json.loads(state_path.read_text()).get("done", 0) if state_path.exists() else 0
    if done >= n:
        print(f"  {split}: {n:,} already done")
        return np.asarray(emb), np.asarray(boxconf[:, :4]), np.asarray(boxconf[:, 4])

    start, fallbacks = time.perf_counter(), 0
    while done < n:
        hi = min(done + CHUNK, n)
        arrays, rows = [], []
        for i in range(done, hi):
            arr = load_working(f"train/{names[i]}")
            if arr is None:
                fallbacks += 1
                continue
            arrays.append(arr)
            rows.append(i)

        if arrays:
            boxes, confs = DETECT(arrays)
            crops = np.empty((len(arrays), SIZE, SIZE, 3), dtype=np.uint8)
            for j, (arr, box, conf) in enumerate(zip(arrays, boxes, confs)):
                use = pad_box(box) if conf >= MIN_CONF else np.array([0, 0, 1, 1], np.float32)
                crops[j] = crop_from_array(arr, use, SIZE)
                boxconf[rows[j]] = [*use, conf]
            emb[np.array(rows)] = embed_tensor(
                torch.from_numpy(crops).permute(0, 3, 1, 2).contiguous())

        done = hi
        emb.flush()
        boxconf.flush()
        state_path.write_text(json.dumps({"done": done}))
        print(f"  {split}: {done:,}/{n:,}  ({time.perf_counter() - start:.0f}s"
              f"{f', {fallbacks} unreadable' if fallbacks else ''})")

    return np.asarray(emb), np.asarray(boxconf[:, :4]), np.asarray(boxconf[:, 4])


frame_cache = CACHE_DIR / f"embeddings_{BACKBONE}_{IMG_SIZE}px.npz"
blob = np.load(frame_cache)
FRAME_EMB = {n: blob[n] for n in SPLITS}
DIM = FRAME_EMB["train"].shape[1]
print("frame stream: loaded", frame_cache.name)

CROP_EMB, BOXES, CONF = {}, {}, {}
for name in SPLITS:
    CROP_EMB[name], BOXES[name], CONF[name] = detect_and_embed(name, DIM)
print(f"\nembedding dimension per stream: {DIM}")

frame stream: loaded embeddings_bioclip_192px.npz
  train: 29,740 already done
  id_val: 2,377 already done
  id_test: 3,420 already done
  ood_val: 4,361 already done

embedding dimension per stream: 512


### 7.1 What the detector found, on everything

The probe in section 6 used 300 images from one split. This is the same question asked of all
39,898, and it is the honest measure of the detector: separation on the splits the model is
scored on, and how tight the boxes actually are.

In [11]:
print(f"{'split':>9} {'AUC animal/empty':>17} {'median side':>12} "
      f"{'full frame':>11} {'mean conf':>10}")
for name in SPLITS:
    empty = LABELS[name] == EMPTY_INDEX
    side = np.maximum(BOXES[name][:, 2] - BOXES[name][:, 0],
                      BOXES[name][:, 3] - BOXES[name][:, 1])
    print(f"{name:>9} {roc_auc(CONF[name], ~empty):>17.3f} {np.median(side):>12.2f} "
          f"{(side >= 0.99).mean():>10.1%} {CONF[name].mean():>10.3f}")

all_side = np.concatenate([np.maximum(BOXES[n][:, 2] - BOXES[n][:, 0],
                                      BOXES[n][:, 3] - BOXES[n][:, 1]) for n in SPLITS])
print(f"\nzoom on the median image: {1 / max(np.median(all_side), 1e-6):.1f}x versus "
      f"using the whole frame")
print("notebook 09's best residual detector: median side 0.93, no label-grounded AUC at all")

    split  AUC animal/empty  median side  full frame  mean conf
    train             0.803         0.50      20.7%      0.630
   id_val             0.786         0.50      21.3%      0.573
  id_test             0.821         0.49      21.8%      0.592
  ood_val             0.812         0.50      18.8%      0.601

zoom on the median image: 2.0x versus using the whole frame
notebook 09's best residual detector: median side 0.93, no label-grounded AUC at all


## 8. Features

Three tabular columns of capture time, plus the detector's own output: box geometry and
confidence. Every one gets the between-camera audit that `seq_len` failed in notebook 06 —
box size encodes how far a camera sits from the trail, which does not transfer, so it is
dropped if it looks like a site fingerprint.

In [12]:
def between_camera_share(values, cameras):
    "Fraction of a column's variance explained by which camera took the shot."
    frame = pd.DataFrame({"v": np.asarray(values, float), "cam": cameras})
    return (frame.groupby("cam").v.transform("mean").var(ddof=0) / frame.v.var(ddof=0))


def detector_features(split):
    box = BOXES[split]
    return pd.DataFrame({
        "box_side": np.maximum(box[:, 2] - box[:, 0], box[:, 3] - box[:, 1]),
        "box_cx": (box[:, 0] + box[:, 2]) / 2,
        "box_cy": (box[:, 1] + box[:, 3]) / 2,
        "conf": CONF[split],
    }, index=SPLITS[split].index)


def time_features(frame):
    dt = pd.to_datetime(frame.datetime, errors="coerce")
    hour = dt.dt.hour + dt.dt.minute / 60.0
    return pd.DataFrame({
        "hour_sin": np.sin(2 * np.pi * hour / 24),
        "hour_cos": np.cos(2 * np.pi * hour / 24),
        "is_night": (((hour >= 19) | (hour <= 5))).astype(float),
    }, index=frame.index).fillna(0.0)


cams_train = SPLITS["train"].location_remapped.to_numpy()
candidates = detector_features("train")
print(f"{'column':>10} {'between-camera variance':>25}  {'verdict':>8}")
KEEP = []
for col in candidates.columns:
    share = between_camera_share(candidates[col], cams_train)
    if share <= 0.50:
        KEEP.append(col)
    print(f"{col:>10} {share:>24.1%}  {'keep' if share <= 0.50 else 'drop':>8}")

META = {n: pd.concat([time_features(f), detector_features(n)[KEEP]], axis=1)
        for n, f in SPLITS.items()}
META_COLS = list(META["train"].columns)
print(f"\n{len(META_COLS)} tabular columns: {META_COLS}")

    column   between-camera variance   verdict
  box_side                    10.4%      keep
    box_cx                     5.0%      keep
    box_cy                    17.2%      keep
      conf                    19.1%      keep

7 tabular columns: ['hour_sin', 'hour_cos', 'is_night', 'box_side', 'box_cx', 'box_cy', 'conf']


## 9. Models

Unchanged from 09 so the only difference between the two notebooks is the detector: three
feature spaces, a linear probe and XGBoost on each, soft votes, and selection by the
one-standard-error rule bootstrapped over cameras.

In [13]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, confusion_matrix, classification_report

SITE_SEPARABLE = ["black-fronted duiker", "empty (false trigger)"]


def honest_macro_f1(truth, preds, exclude=SITE_SEPARABLE):
    per_class = f1_score(truth, preds, average=None, labels=range(N_CLASSES), zero_division=0)
    keep = [i for i, name in enumerate(CLASS_NAMES) if name not in exclude]
    return per_class[keep].mean()


STREAMS = {
    "frame": lambda s: FRAME_EMB[s],
    "crop":  lambda s: CROP_EMB[s],
    "both":  lambda s: np.hstack([FRAME_EMB[s], CROP_EMB[s]]),
}

PCAS = {}
for name, get in STREAMS.items():
    p = PCA(n_components=min(PCA_DIMS, get("train").shape[1]), random_state=SEED)
    p.fit(get("train")[FIT_MASK])
    PCAS[name] = p
    print(f"{name:>6}: {get('train').shape[1]:>5} dims -> PCA {p.n_components_}, "
          f"{p.explained_variance_ratio_.sum():.1%} variance")


def design(split, mask=None, stream="both", compress=False, with_meta=False):
    block = STREAMS[stream](split)
    meta = META[split].to_numpy()
    if mask is not None:
        block, meta = block[mask], meta[mask]
    if compress:
        block = PCAS[stream].transform(block)
    return np.hstack([block, meta]) if with_meta else block


XGB_PARAMS = dict(n_estimators=2000, learning_rate=0.05, max_depth=6, min_child_weight=2.0,
                  subsample=0.8, colsample_bytree=0.5, reg_lambda=2.0, reg_alpha=0.1)


def fit_linear(cfg):
    model = make_pipeline(StandardScaler(),
                          LogisticRegression(max_iter=3000, C=1.0,
                                             class_weight="balanced"))
    model.fit(design("train", FIT_MASK, **cfg), y_fit)
    return model


def fit_boosted(cfg):
    X = design("train", FIT_MASK, **cfg)
    w = compute_sample_weight("balanced", y_fit)
    probe = XGBClassifier(objective="multi:softprob", eval_metric="mlogloss",
                          early_stopping_rounds=50, random_state=SEED, n_jobs=-1,
                          verbosity=0, **XGB_PARAMS, **ACCEL)
    probe.fit(X[INNER_FIT], y_fit[INNER_FIT], sample_weight=w[INNER_FIT],
              eval_set=[(X[INNER_ES], y_fit[INNER_ES])],
              sample_weight_eval_set=[compute_sample_weight("balanced", y_fit[INNER_ES])],
              verbose=False)
    rounds = int(round((int(probe.best_iteration) + 1) / (1 - ES_FRAC)))
    final = XGBClassifier(objective="multi:softprob", eval_metric="mlogloss",
                          random_state=SEED, n_jobs=-1, verbosity=0,
                          **dict(XGB_PARAMS, n_estimators=rounds), **ACCEL)
    final.fit(X, y_fit, sample_weight=w, verbose=False)
    return final


CONFIGS = {}
for stream in STREAMS:
    CONFIGS[f"linear ({stream} full)"] = dict(stream=stream, compress=False, with_meta=False)
    CONFIGS[f"linear ({stream}+meta)"] = dict(stream=stream, compress=True, with_meta=True)
    CONFIGS[f"xgboost ({stream}+meta)"] = dict(stream=stream, compress=True, with_meta=True)

MODELS = {}
print(f"\n{'model':>26} {'columns':>8} {'sel acc':>8} {'sel honest':>11} {'time':>7}")
for label, cfg in CONFIGS.items():
    start = time.perf_counter()
    model = fit_linear(cfg) if label.startswith("linear") else fit_boosted(cfg)
    MODELS[label] = dict(model=model, cfg=cfg, complexity=1)
    preds = model.predict(design("train", SEL_MASK, **cfg))
    print(f"{label:>26} {design('train', FIT_MASK, **cfg).shape[1]:>8,} "
          f"{(preds == y_sel).mean():>8.3f} {honest_macro_f1(y_sel, preds):>11.3f} "
          f"{time.perf_counter() - start:>6.0f}s")

 frame:   512 dims -> PCA 128, 90.0% variance
  crop:   512 dims -> PCA 128, 87.6% variance
  both:  1024 dims -> PCA 128, 81.8% variance

                     model  columns  sel acc  sel honest    time
       linear (frame full)      512    0.566       0.455     56s
       linear (frame+meta)      135    0.658       0.565     12s
      xgboost (frame+meta)      135    0.650       0.576     12s
        linear (crop full)      512    0.695       0.620     72s
        linear (crop+meta)      135    0.719       0.653     11s
       xgboost (crop+meta)      135    0.711       0.651     12s
        linear (both full)    1,024    0.674       0.574     81s
        linear (both+meta)      135    0.711       0.626     12s
       xgboost (both+meta)      135    0.713       0.640     14s


In [14]:
_CACHE = {}


def probs_for(label, split, mask=None):
    key = (label, split, mask is not None)
    if key not in _CACHE:
        entry = MODELS[label]
        if entry.get("members"):
            p = np.mean([probs_for(m, split, mask) for m in entry["members"]], axis=0)
        else:
            p = entry["model"].predict_proba(design(split, mask, **entry["cfg"]))
        _CACHE[key] = p
    return _CACHE[key]


sel_now = {k: honest_macro_f1(y_sel, probs_for(k, "train", SEL_MASK).argmax(axis=1))
           for k in list(MODELS)}
for stream in STREAMS:
    MODELS[f"vote ({stream})"] = dict(members=[k for k in sel_now if f"({stream}" in k],
                                      cfg=None, complexity=0)
MODELS["vote (everything)"] = dict(members=list(sel_now), cfg=None, complexity=0)

SEL_CAMS = train_frame[SEL_MASK].location_remapped.to_numpy()


def bootstrap_selection(label, draws=200, seed=SEED):
    "Honest macro F1 on the selection cameras, resampled over CAMERAS not images."
    preds = probs_for(label, "train", SEL_MASK).argmax(axis=1)
    rng = np.random.default_rng(seed)
    cams = np.unique(SEL_CAMS)
    rows = {c: np.flatnonzero(SEL_CAMS == c) for c in cams}
    scores = np.empty(draws)
    for d in range(draws):
        pick = np.concatenate([rows[c] for c in rng.choice(cams, len(cams), replace=True)])
        scores[d] = honest_macro_f1(y_sel[pick], preds[pick])
    return honest_macro_f1(y_sel, preds), scores.std()


SEL, SE = {}, {}
for label in MODELS:
    SEL[label], SE[label] = bootstrap_selection(label)

top = max(SEL, key=SEL.get)
cutoff = SEL[top] - SE[top]
tied = [k for k in SEL if SEL[k] >= cutoff]
BEST = min(tied, key=lambda k: (MODELS[k]["complexity"], -SEL[k]))

print(f"{'model':>26} {'selection':>10} {'+/- 1 se':>9} {'within 1 se':>12}")
for label in sorted(SEL, key=SEL.get, reverse=True):
    print(f"{label:>26} {SEL[label]:>10.3f} {SE[label]:>9.3f} "
          f"{'yes' if label in tied else '':>12}")
print(f"\nselected under the one-se rule: {BEST}")

                     model  selection  +/- 1 se  within 1 se
        linear (crop+meta)      0.653     0.043          yes
       xgboost (crop+meta)      0.651     0.047          yes
               vote (crop)      0.651     0.042          yes
         vote (everything)      0.647     0.040          yes
       xgboost (both+meta)      0.640     0.053          yes
        linear (both+meta)      0.626     0.041          yes
               vote (both)      0.623     0.043          yes
        linear (crop full)      0.620     0.043          yes
      xgboost (frame+meta)      0.576     0.049             
        linear (both full)      0.574     0.041             
       linear (frame+meta)      0.565     0.041             
              vote (frame)      0.539     0.036             
       linear (frame full)      0.455     0.041             

selected under the one-se rule: vote (crop)


## 10. Results on the official unseen-camera split

In [ ]:
def aggregate(probs, frame, how="mean", conf=None):
    """Turn per-image probabilities into one answer per burst.

    mean       average the probabilities across the burst - what this notebook used
    max        the single most confident (frame, class) in the burst wins - the rule
               both iWildCam 2022 write-ups settled on, and worth +0.041 honest in
               notebook 12
    best frame the frame whose detection confidence was highest decides
    """
    out = np.empty(len(frame), dtype=int)
    seq = frame.seq_id.to_numpy()
    for s in np.unique(seq):
        rows = np.flatnonzero(seq == s)
        if how == "mean":
            out[rows] = probs[rows].mean(axis=0).argmax()
        elif how == "max":
            out[rows] = np.unravel_index(probs[rows].argmax(), probs[rows].shape)[1]
        else:
            out[rows] = probs[rows[conf[rows].argmax()]].argmax()
    return out


RULES = ("mean", "max", "best frame")

print(f"{'split':>9} {'accuracy':>10} {'macro F1':>10} {'honest':>8}")
for split in ("id_test", "ood_val"):
    probs = probs_for(BEST, split)
    truth, fp = LABELS[split], probs.argmax(axis=1)
    print(f"{split:>9} {(fp == truth).mean():>10.3f} "
          f"{f1_score(truth, fp, average='macro'):>10.3f} "
          f"{honest_macro_f1(truth, fp):>8.3f}")

ood_probs = probs_for(BEST, "ood_val")
ood_truth = LABELS["ood_val"]
ood_preds = ood_probs.argmax(axis=1)

print(f"\n{'sequence rule':>16} {'accuracy':>10} {'macro F1':>10} {'honest':>8} "
      f"{'vs mean':>9}")
seq_scores = {}
for how in RULES:
    sp = aggregate(ood_probs, SPLITS["ood_val"], how, CONF["ood_val"])
    seq_scores[how] = (sp, (sp == ood_truth).mean(),
                       f1_score(ood_truth, sp, average="macro"),
                       honest_macro_f1(ood_truth, sp))
    note = ("  <- was used here" if how == "mean" else
            "  <- theirs" if how == "max" else "")
    delta = seq_scores[how][3] - seq_scores["mean"][3]
    print(f"{how:>16} {seq_scores[how][1]:>10.3f} {seq_scores[how][2]:>10.3f} "
          f"{seq_scores[how][3]:>8.3f} {delta:>+9.3f}{note}")

BEST_RULE = max(RULES, key=lambda h: seq_scores[h][3])
ood_seq = seq_scores[BEST_RULE][0]
print(f"\nbest sequence rule: {BEST_RULE}")

print()
print(classification_report(ood_truth, ood_preds, target_names=CLASS_NAMES,
                            digits=3, zero_division=0))

In [16]:
NB08 = {"empty (false trigger)": 0.964, "ocellated turkey": 0.584, "great curassow": 0.720,
        "white-lipped peccary": 0.597, "black-fronted duiker": 0.929, "agouti": 0.284,
        "ocelot": 0.738, "red brocket deer": 0.302, "puma": 0.320, "gray fox": 0.372}
NB09 = {"empty (false trigger)": 0.955, "ocellated turkey": 0.571, "great curassow": 0.728,
        "white-lipped peccary": 0.488, "black-fronted duiker": 0.967, "agouti": 0.374,
        "ocelot": 0.741, "red brocket deer": 0.388, "puma": 0.350, "gray fox": 0.439}

per_class = f1_score(ood_truth, ood_preds, average=None, labels=range(N_CLASSES),
                     zero_division=0)
print(f"{'class':<24} {'support':>8} {'nb08':>7} {'nb09':>7} {'this':>7} {'vs 09':>8}")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:<24} {int((ood_truth == i).sum()):>8,} {NB08[name]:>7.3f} "
          f"{NB09[name]:>7.3f} {per_class[i]:>7.3f} {per_class[i] - NB09[name]:>+8.3f}")

print(f"\n{'model':>26} {'ood acc':>9} {'ood honest':>11}")
for label in sorted(MODELS, key=lambda k: -honest_macro_f1(
        ood_truth, probs_for(k, "ood_val").argmax(axis=1))):
    p = probs_for(label, "ood_val").argmax(axis=1)
    star = "  <- selected" if label == BEST else ""
    print(f"{label:>26} {(p == ood_truth).mean():>9.3f} "
          f"{honest_macro_f1(ood_truth, p):>11.3f}{star}")

class                     support    nb08    nb09    this    vs 09
empty (false trigger)         600   0.964   0.955   0.948   -0.007
ocellated turkey              600   0.584   0.571   0.715   +0.144
great curassow                600   0.720   0.728   0.818   +0.090
white-lipped peccary          600   0.597   0.488   0.758   +0.270
black-fronted duiker          600   0.929   0.967   0.889   -0.078
agouti                        210   0.284   0.374   0.399   +0.025
ocelot                        470   0.738   0.741   0.770   +0.029
red brocket deer              130   0.302   0.388   0.450   +0.062
puma                          241   0.320   0.350   0.383   +0.033
gray fox                      310   0.372   0.439   0.490   +0.051

                     model   ood acc  ood honest
         vote (everything)     0.739       0.609
               vote (crop)     0.725       0.598  <- selected
       xgboost (crop+meta)     0.696       0.582
               vote (both)     0.719       0.582
    

In [ ]:
import itertools

ood_scores = {k: honest_macro_f1(ood_truth, probs_for(k, "ood_val").argmax(axis=1))
              for k in MODELS}
pairs = [(a, b) for a, b in itertools.combinations(SEL, 2)
         if SEL[a] != SEL[b] and ood_scores[a] != ood_scores[b]]
agree = sum(1 for a, b in pairs if (SEL[a] - SEL[b]) * (ood_scores[a] - ood_scores[b]) > 0)
share = agree / len(pairs) if pairs else float("nan")
print(f"rank agreement: {agree}/{len(pairs)} ordered pairs ({share:.0%}) "
      f"({'informative' if share >= 0.7 else 'NOT TRACKING THE REAL TASK'})")
print(f"the one-se rule was worth {ood_scores[BEST] - ood_scores[top]:+.3f}\n")

print(f"{'approach':<54} {'accuracy':>10} {'macro F1':>10} {'honest':>8}")
print(f"{'ResNet-18 fine-tuned, 192px (v3)':<54} {0.637:>10.3f} {0.543:>10.3f} {0.446:>8.3f}")
print(f"{'EffNetV2-S + XGBoost (nb 07)':<54} {0.581:>10.3f} {0.512:>10.3f} {0.437:>8.3f}")
print(f"{'BioCLIP, full frame (nb 08)':<54} {0.653:>10.3f} {0.581:>10.3f} {0.490:>8.3f}")
print(f"{'BioCLIP + residual crop, thumbnails (nb 09)':<54} "
      f"{0.672:>10.3f} {0.600:>10.3f} {0.510:>8.3f}")
print(f"{'   ...per sequence':<54} {0.701:>10.3f} {0.636:>10.3f} {0.552:>8.3f}")
print()
print(f"{f'BioCLIP + {DETECTOR} crop, {BEST}':<54} "
      f"{(ood_preds == ood_truth).mean():>10.3f} "
      f"{f1_score(ood_truth, ood_preds, average='macro'):>10.3f} "
      f"{honest_macro_f1(ood_truth, ood_preds):>8.3f}")
print(f"{'   ...per sequence, ' + BEST_RULE:<54} {(ood_seq == ood_truth).mean():>10.3f} "
      f"{f1_score(ood_truth, ood_seq, average='macro'):>10.3f} "
      f"{honest_macro_f1(ood_truth, ood_seq):>8.3f}")